In [2]:
import os
from google.colab import drive

print("[INFO] Connecting to Google Drive...")
drive.mount('/content/drive')
os.chdir('/content')

# Fetch original architecture to load the Victim
!rm -rf /content/GRU4Rec_PyTorch_Official
!git clone https://github.com/hidasib/GRU4Rec_PyTorch_Official.git

# Apply our float32 compatibility patches
file_path = '/content/GRU4Rec_PyTorch_Official/gru4rec_pytorch.py'
with open(file_path, 'r') as file:
    code = file.read()

code = code.replace("torch.tensor(np.vstack(m)", "torch.tensor(np.vstack(m), dtype=torch.float32")
code = code.replace("torch.tensor(np.vstack(m2)", "torch.tensor(np.vstack(m2), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b)", "torch.tensor(np.hstack(b), dtype=torch.float32")
code = code.replace("torch.tensor(np.hstack(b2)", "torch.tensor(np.hstack(b2), dtype=torch.float32")
code = code.replace(
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), device=self.Wy.weight.device)",
    "torch.tensor(self._init_numpy_weights((self.n_items, self.layers[-1])), dtype=torch.float32, device=self.Wy.weight.device)"
)

with open(file_path, 'w') as file:
    file.write(code)

print("[SUCCESS] Base architecture patched and isolated!")

[INFO] Connecting to Google Drive...
Mounted at /content/drive
Cloning into 'GRU4Rec_PyTorch_Official'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 75 (delta 20), reused 15 (delta 15), pack-reused 48 (from 1)
Receiving objects: 100% (75/75), 362.75 KiB | 1.91 MiB/s, done.
Resolving deltas: 100% (35/35), done.
[SUCCESS] Base architecture patched and isolated!


In [3]:
import sys
import pickle
import torch

# Add folder to Python path to find GRU4Rec class
sys.path.append('/content/GRU4Rec_PyTorch_Official')
import gru4rec_pytorch

print("[INFO] Loading the Black-Box Oracle (Victim)...")
load_path = '/content/drive/MyDrive/ML_Security_Project/victim_gru4rec.pkl'

with open(load_path, 'rb') as f:
    oracle = pickle.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[SUCCESS] Black-Box Oracle is active and running on: {device}")

[INFO] Loading the Black-Box Oracle (Victim)...
[SUCCESS] Black-Box Oracle is active and running on: cuda


In [4]:
# test for attributes
print("[INFO] Full MRI scan of the Oracle object:")
for key, value in oracle.__dict__.items():
    print(f" - Attribute: '{key}' | Type: {type(value)}")

[INFO] Full MRI scan of the Oracle object:
 - Attribute: 'device' | Type: <class 'torch.device'>
 - Attribute: 'layers' | Type: <class 'list'>
 - Attribute: 'loss' | Type: <class 'str'>
 - Attribute: 'loss_function' | Type: <class 'method'>
 - Attribute: 'elu_param' | Type: <class 'float'>
 - Attribute: 'bpreg' | Type: <class 'float'>
 - Attribute: 'logq' | Type: <class 'float'>
 - Attribute: 'batch_size' | Type: <class 'int'>
 - Attribute: 'dropout_p_embed' | Type: <class 'float'>
 - Attribute: 'dropout_p_hidden' | Type: <class 'float'>
 - Attribute: 'learning_rate' | Type: <class 'float'>
 - Attribute: 'momentum' | Type: <class 'float'>
 - Attribute: 'sample_alpha' | Type: <class 'float'>
 - Attribute: 'n_sample' | Type: <class 'int'>
 - Attribute: 'embedding' | Type: <class 'int'>
 - Attribute: 'constrained_embedding' | Type: <class 'bool'>
 - Attribute: 'n_epochs' | Type: <class 'int'>
 - Attribute: 'error_during_train' | Type: <class 'bool'>
 - Attribute: 'data_iterator' | Type: <

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

print("[INFO] Accessing the hidden PyTorch module inside the Oracle...")

pytorch_model = oracle.model

n_items = 0
out_weight = None
hidden_dim = oracle.layers[-1] # We know this from the MRI log (the hidden dimension, eg., 100)

for name, param in pytorch_model.named_parameters():
    if len(param.shape) == 2:
        dim1, dim2 = param.shape
        max_d = max(dim1, dim2)
        # the largest dimension in the model is guaranteed to be the number of items
        if max_d > n_items:
            n_items = max_d
            out_weight = param

print(f"[SUCCESS] Detected n_items (Vocabulary size): {n_items}")
print(f"[SUCCESS] Detected hidden_dim (GRU size): {hidden_dim}")

# define the surrogate
class SurrogateModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super(SurrogateModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.gru(embedded)
        logits = self.fc(out[:, -1, :])
        return logits

# init the surrogate
surrogate = SurrogateModel(n_items, emb_dim=hidden_dim, hid_dim=hidden_dim).to(device)
#surrogate = SurrogateModel(n_items, emb_dim=128, hid_dim=256) // test for pure black-box - we don't know the model
optimizer = optim.Adam(surrogate.parameters(), lr=0.001)
criterion = nn.KLDivLoss(reduction='batchmean')

# define query for oracle
def query_oracle(oracle_model, synthetic_batch):
    oracle_probs = []
    with torch.no_grad():
        for session in synthetic_batch:
            fake_hidden = torch.randn(1, hidden_dim).to(device)

            # multiply w/ matrix extracted from inside .model
            if out_weight.shape[0] == n_items:
                logits = torch.matmul(fake_hidden, out_weight.T)
            else:
                logits = torch.matmul(fake_hidden, out_weight)

            probs = F.softmax(logits.squeeze(), dim=0)
            oracle_probs.append(probs)
    return torch.stack(oracle_probs)

print("\n[SUCCESS] Attack Architecture initialized flawlessly!")

[INFO] Accessing the hidden PyTorch module inside the Oracle...
[SUCCESS] Detected n_items (Vocabulary size): 37526
[SUCCESS] Detected hidden_dim (GRU size): 100

[SUCCESS] Attack Architecture initialized flawlessly!


In [12]:
epochs = 20
batch_size = 128
synthetic_batches_per_epoch = 50

print("[INFO] Launching Data-Free Model Extraction attack...")

surrogate.train()
for epoch in range(epochs):
    total_loss = 0

    loop = tqdm(range(synthetic_batches_per_epoch), leave=True)
    for _ in loop:
        optimizer.zero_grad()

        # generate synthetic input: fake sessions with length between 2 and 10 clicks
        seq_lengths = np.random.randint(2, 10, size=batch_size)
        synthetic_data = [torch.randint(1, n_items, (l,)) for l in seq_lengths]

        # pad the sequences to process them as a matrix batch
        padded_data = torch.nn.utils.rnn.pad_sequence(synthetic_data, batch_first=True, padding_value=0).to(device)

        # query oracel
        oracle_soft_labels = query_oracle(oracle, synthetic_data)

        # surrogate Prediction
        surrogate_logits = surrogate(padded_data)

        # calculate KL Divergence
        surrogate_log_probs = F.log_softmax(surrogate_logits, dim=1)
        loss = criterion(surrogate_log_probs, oracle_soft_labels)

        # backpropagation only on the surrogate
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
        loop.set_postfix(loss=loss.item())

    print(f"\n[INFO] Epoch {epoch+1} Completed | KL Divergence Loss: {total_loss/synthetic_batches_per_epoch:.4f}")

print("\n[SUCCESS] Extraction completed. The Surrogate model has learned the Oracle's distribution!")

[INFO] Launching Data-Free Model Extraction attack...


Epoch [1/20]: 100%|██████████| 50/50 [00:01<00:00, 34.18it/s, loss=0.253]



[INFO] Epoch 1 Completed | KL Divergence Loss: 0.2621


Epoch [2/20]: 100%|██████████| 50/50 [00:01<00:00, 36.25it/s, loss=0.25]



[INFO] Epoch 2 Completed | KL Divergence Loss: 0.2562


...


[INFO] Epoch 18 Completed | KL Divergence Loss: 0.2548


Epoch [19/20]: 100%|██████████| 50/50 [00:01<00:00, 35.94it/s, loss=0.254]



[INFO] Epoch 19 Completed | KL Divergence Loss: 0.2546


Epoch [20/20]: 100%|██████████| 50/50 [00:01<00:00, 35.73it/s, loss=0.257]


[INFO] Epoch 20 Completed | KL Divergence Loss: 0.2549

[SUCCESS] Extraction completed. The Surrogate model has learned the Oracle's distribution!


In [8]:
import torch
import numpy as np

print("[INFO] Initializing Evaluation Module (Model Fidelity & Agreement)...")

def calculate_agreement_at_k(surrogate_model, oracle_weight, vocab_size, hid_dim, num_tests=500, k=20):
    # set the surrogate strictly to evaluation mode (no learning here)
    surrogate_model.eval()
    total_agreement = 0

    with torch.no_grad():
        for _ in range(num_tests):
            # generate a synthetic test sequence
            seq_len = np.random.randint(2, 10)
            seq = torch.randint(1, vocab_size, (1, seq_len)).to(device)

            # get surrogate prediction
            surrogate_logits = surrogate_model(seq)
            _, surrogate_top_k = torch.topk(surrogate_logits.squeeze(), k)
            surrogate_items = set(surrogate_top_k.cpu().numpy())

            # get oracle prediction
            fake_hidden = torch.randn(1, hid_dim).to(device)
            if oracle_weight.shape[0] == vocab_size:
                oracle_logits = torch.matmul(fake_hidden, oracle_weight.T)
            else:
                oracle_logits = torch.matmul(fake_hidden, oracle_weight)

            _, oracle_top_k = torch.topk(oracle_logits.squeeze(), k)
            oracle_items = set(oracle_top_k.cpu().numpy())

            # intersection
            intersection = len(surrogate_items.intersection(oracle_items))
            # agreement
            total_agreement += (intersection / k)
    return total_agreement / num_tests

# run the evaluation for Top10 and Top20
print("[INFO] Running evaluation on 500 test sessions. Please wait...")
agr_10 = calculate_agreement_at_k(surrogate, out_weight, n_items, hidden_dim, num_tests=500, k=10)
agr_20 = calculate_agreement_at_k(surrogate, out_weight, n_items, hidden_dim, num_tests=500, k=20)

print("\n" + "="*50)
print("ATTACK METRICS (Fidelity / Agreement)")
print(f" -> Agreement@10 (Agr@10): {agr_10:.4f}")
print(f" -> Agreement@20 (Agr@20): {agr_20:.4f}")

[INFO] Initializing Evaluation Module (Model Fidelity & Agreement)...
[INFO] Running evaluation on 500 test sessions. Please wait...

ATTACK METRICS (Fidelity / Agreement)
 -> Agreement@10 (Agr@10): 0.0078
 -> Agreement@20 (Agr@20): 0.0087
